### Silver Transformation

#### Parameters and Environment Setup

In [ ]:
today_date = '2026-02-25'

In [1]:
# Parameters
bronze_path = "abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Tables/dbo/tblsales_bronze"

from pyspark.sql.functions import col, sha2, concat_ws, lit, current_date, monotonically_increasing_id, to_date
from datetime import date

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 12, Finished, Available, Finished, False)

#### Load Data and Apply the 3 DQ Checks


In [2]:
# Load today's batch
df_bronze = spark.read.format('delta').load(bronze_path).filter(col('processing_date') == str(today_date))

# Applying ONLY the 3 requested checks:
# 1. No null property_id, rooms (bedrooms), or price.
# 2. Positive price only.
# 3. Valid month format (ensures the date transformation in Bronze didn't return Null).
df_cleaned = df_bronze.filter(
    (col("property_id").isNotNull()) & 
    (col("bedrooms").isNotNull()) & 
    (col("price").isNotNull()) &
    (col("price") > 0) & 
    (col("month").isNotNull())
).dropDuplicates()
df_cleaned

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 13, Finished, Available, Finished, False)

DataFrame[property_id: string, month: string, price: double, area: int, bedrooms: int, bathrooms: int, stories: int, mainroad: string, guestroom: string, basement: string, hotwaterheating: string, airconditioning: string, parking: int, prefarea: string, furnishingstatus: string, processing_date: string]

#### Splitting the Dataset

In [4]:
### 1. House Price Time Series (Fact-like data)
# This keeps the price history for every property id and month
df_price_timeseries = df_cleaned.select(
    "property_id", 
    "month",  
    "price", 
    "processing_date"
)

# 2. Houses Master Data (Dimension-like data for SCD Type 2)
# This contains the characteristics we want to track over time
df_houses = df_cleaned.select(
    "property_id", 
    "area", 
    "bedrooms", 
    "bathrooms", 
    "stories", 
    "mainroad", 
    "guestroom", 
    "basement", 
    "hotwaterheating", 
    "airconditioning", 
    "parking", 
    "prefarea", 
    "furnishingstatus"
).dropDuplicates(["property_id"]) # Get the latest state of the house characteristics for this batch

display(df_houses)

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6ec3857-f7c4-412b-8e2c-3b3c4b9c257e)

### Hashing & Surrogate Key Generation

In [3]:
# --- 1. Preparation for Houses (Master Data) ---
change_track_cols = [
    "area", "bedrooms", "bathrooms", "stories", "mainroad", 
    "guestroom", "basement", "hotwaterheating", "airconditioning", 
    "parking", "prefarea", "furnishingstatus"
]

df_houses_source = df_cleaned.select(
    "property_id", "area", "bedrooms", "bathrooms", "stories", 
    "mainroad", "guestroom", "basement", "hotwaterheating", 
    "airconditioning", "parking", "prefarea", "furnishingstatus"
).dropDuplicates(["property_id"]).withColumn(
    "row_hash", sha2(concat_ws("||", *change_track_cols), 256)
).withColumn(
    "house_sk", monotonically_increasing_id()
)

# Create Temp View for MERGE
df_houses_source.createOrReplaceTempView('t_silver_houses_new_data')

# --- 2. Preparation for HousePriceTimeSeries ---
df_timeseries = df_cleaned.select(
    "property_id", "month", "price", "processing_date"
)
df_timeseries.createOrReplaceTempView('t_silver_ts_new_data')

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 15, Finished, Available, Finished, False)

#### Create Silver Delta Tables

In [4]:
path_silver_houses = 'abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Tables/dbo/silver_houses'

try:
    spark.read.format('delta').load(path_silver_houses).createOrReplaceTempView('t_silver_houses_current')
except:
    v_create_houses = f"""CREATE TABLE IF NOT EXISTS silver_houses (
            house_sk LONG, property_id string, area int, bedrooms int, bathrooms int, stories int,
            mainroad string, guestroom string, basement string, hotwaterheating string,
            airconditioning string, parking int, prefarea string, furnishingstatus string,
            row_hash string, IsCurrent boolean, ValidFrom date, ValidTo date
    ) USING DELTA"""
    spark.sql(v_create_houses)
    spark.read.format('delta').load(path_silver_houses).createOrReplaceTempView('t_silver_houses_current')

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 16, Finished, Available, Finished, False)

#### SCD Type 2 - Part 1: Expire Old Versions

In [5]:
# Expire records that have changed
sql_expire_houses = """
MERGE INTO silver_houses AS target
USING t_silver_houses_new_data AS source
ON target.property_id = source.property_id AND target.IsCurrent = true
WHEN MATCHED AND target.row_hash <> source.row_hash THEN
    UPDATE SET target.IsCurrent = false, target.ValidTo = current_date()
"""
spark.sql(sql_expire_houses)

# Insert the new versions (or brand new records)
df_to_insert_houses = spark.sql("""
    SELECT 
        s.*, true as IsCurrent, current_date() as ValidFrom, cast(null as date) as ValidTo
    FROM t_silver_houses_new_data s
    LEFT JOIN silver_houses t 
        ON s.property_id = t.property_id AND t.IsCurrent = true AND s.row_hash = t.row_hash
    WHERE t.property_id IS NULL
""")
df_to_insert_houses.write.format("delta").mode("append").saveAsTable("silver_houses")

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 17, Finished, Available, Finished, False)

#### SCD Type 2 - Part 2: Insert New/Updated Versions

In [6]:
path_silver_ts = 'abfss://house_price_DEV@onelake.dfs.fabric.microsoft.com/houseprice_LH.Lakehouse/Tables/dbo/silver_house_price_timeseries'

try:
    spark.read.format('delta').load(path_silver_ts).createOrReplaceTempView('t_silver_ts_current')
except:
    v_create_ts = """CREATE TABLE IF NOT EXISTS silver_house_price_timeseries (
            property_id string, month string, price double, processing_date date
    ) USING DELTA"""
    spark.sql(v_create_ts)

# MERGE logic for TimeSeries (Upsert)
sql_merge_ts = """
MERGE INTO silver_house_price_timeseries AS target
USING t_silver_ts_new_data AS source
ON target.property_id = source.property_id AND target.month = source.month
WHEN MATCHED THEN
    UPDATE SET 
        target.price = source.price, 
        target.processing_date = source.processing_date
WHEN NOT MATCHED THEN
    INSERT (property_id, month,  price, processing_date)
    VALUES (source.property_id, source.month, source.price, source.processing_date)
"""
spark.sql(sql_merge_ts)

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 18, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### Time Series Merge

In [ ]:
# MERGE logic for TimeSeries (Upsert)
sql_merge_ts = """
MERGE INTO silver_house_price_timeseries AS target
USING t_silver_ts_new_data AS source
ON target.property_id = source.property_id AND target.month = source.month
WHEN MATCHED THEN
    UPDATE SET 
        target.price = source.price, 
        target.processing_date = source.processing_date
WHEN NOT MATCHED THEN
    INSERT (property_id, month,  price, processing_date)
    VALUES (source.property_id, source.month, source.price, source.processing_date)
"""
spark.sql(sql_merge_ts)

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 19, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [10]:
%%html
select * from silver_houses

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 20, Finished, Available, Finished, False)

<Spark SQL result set with 543 rows and 18 fields>

In [12]:
%%html
Select * from silver_house_price_timeseries

StatementMeta(, 2c3af129-3403-403a-8160-b9ef36434381, 21, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 4 fields>